# EQO notebook: FTQC → IQM experiment

This is the executable hardware-integration example: **OpenQASM input → FTQC preparation → IQM routing, submission, and result collection**. It will run only when EQO has an admitted `quantum-backend` worker. Credentials stay worker-local; this notebook never reads or sends an IQM token.

In [ ]:
import os

from eqo import EQOClient, render_artifact, render_run

eqo = EQOClient.connect(os.environ.get("EQO_ENDPOINT", "http://127.0.0.1:8080"))
eqo.health()

## Confirm the admitted IQM worker

For a safe dry run, start EQO with `eqo local up --iqm-simulation`. For an internal IQM run, start the isolated IQM worker with its endpoint, device alias, and worker-local token. The worker inventory is the source of truth; no notebook setting can substitute a backend worker.

In [ ]:
workers = eqo.workers()
iqm_workers = [
    worker for worker in workers
    if "quantum-backend" in worker.get("metadata", {}).get("execution_classes", [])
]
if not iqm_workers:
    raise RuntimeError(
        "No admitted IQM quantum-backend worker is ready. Start --iqm-simulation or the internal IQM worker, then rerun this cell."
)
iqm_workers

## Create a measured Bell circuit

The input is an EQO artifact. FTQC reads it only inside the reviewed OCI container and emits FTQC MLIR plus IQM circuit and preparation-report artifacts before the worker reaches IQM.

In [ ]:
bell_qasm = """OPENQASM 3.0;
include \"stdgates.inc\";

qubit[2] q;
bit[2] result;
h q[0];
cx q[0], q[1];
result[0] = measure q[0];
result[1] = measure q[1];
"""
input_circuit = eqo.artifacts.create_input(
    "qhpc.quantum-circuit@1", bell_qasm, name="bell-qpu.qasm"
)
input_circuit.metadata

## Submit only after reviewing the worker inventory

The published workflow uses the configured internal device alias and `secret://env/IQM_TOKEN` only inside the worker. Inspect `device.provider` in the resulting receipt: `simulated-iqm` is explicitly a simulation, while `iqm` identifies a hardware provider. Treat those cases differently in analysis and reporting.

In [ ]:
workflow = next((item for item in eqo.workflows.list() if item["id"] == "ftqc-iqm-bell-execution"), None)
if workflow is None:
    raise RuntimeError("ftqc-iqm-bell-execution is not published by this EQO profile.")
run = eqo.workflows.submit(
    workflow["id"], workflow["version"], inputs={"circuit": input_circuit.id}
)
render_run(run)

In [ ]:
completed = run.wait(timeout=2100)
if completed.state != "succeeded":
    raise RuntimeError(f"IQM experiment ended in {completed.state}; inspect render_run(completed).")
for artifact_type in (
    "qhpc.ftqc-mlir@1",
    "qhpc.iqm-circuit@1",
    "qhpc.ftqc-iqm-preparation-report@1",
    "qhpc.iqm-routed-layout@1",
    "qhpc.iqm-job-receipt@1",
    "qhpc.iqm-raw-counts@1",
    "qhpc.ftqc-logical-result@1",
):
    display(render_artifact(completed.artifacts.by_type(artifact_type)))

The `qhpc.iqm-job-receipt@1` artifact records the device provider, alias, computer, calibration, and job identity. Preserve it with the counts and logical result; a `device.provider` value of `simulated-iqm` is not hardware evidence.